# CYP51 MM/GBSA — corrected workflow for CAL-028 and CAL-015

Uses the completed 50 ns trajectories in Drive, AmberTools 26 (`MMPBSA.py`), and 5 resumable 10 ns blocks per system.

The generated driver removes the unsupported `strip_mdcrd`, preserves completed blocks, and produces reports separately.

**Execution status:** CAL-028 and CAL-015 smoke tests were reported successful. Only CAL-028 Block 1 was reported complete; full 50 ns MM/GBSA remains pending.


## 1. Mount Drive and verify original trajectories

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/CYP51_MD_animation_sources')
for system in ['CAL028','CAL015']:
    folder=ROOT/system
    top=folder/'system.prmtop'
    dcds=[folder/f'ns_{i:03d}.dcd' for i in range(1,51)]
    missing=[str(p) for p in [top]+dcds if not p.is_file()]
    if missing: raise FileNotFoundError(f'{system}: missing {missing[:5]}')
    size=(sum(p.stat().st_size for p in dcds)+top.stat().st_size)/(1024**3)
    print(f'{system}: 50 DCD + topology verified, {size:.2f} GiB')
print('All original files are present; they will not be altered.')

## 2. Install AmberTools 26 in isolated micromamba environment

First installation downloads packages from conda-forge. Subsequent cells call its dedicated Python interpreter and Amber executables. If the Colab runtime resets, rerun this installation cell.

In [ ]:
from pathlib import Path
import urllib.request, tarfile, subprocess, os
MMROOT=Path('/content/mmgbsa_mamba')
EXE=Path('/content/mmgbsa_tools/bin/micromamba')
if not EXE.is_file():
    Path('/content/mmgbsa_tools').mkdir(exist_ok=True)
    archive=Path('/content/micromamba.tar.bz2')
    urllib.request.urlretrieve('https://micro.mamba.pm/api/micromamba/linux-64/latest',str(archive))
    with tarfile.open(archive,'r:bz2') as tar:
        member=tar.getmember('bin/micromamba')
        tar.extract(member,'/content/mmgbsa_tools',filter='data')
    EXE.chmod(0o755)
ENV=MMROOT/'envs'/'amber26'
if not (ENV/'bin'/'MMPBSA.py').exists():
    subprocess.run([str(EXE),'create','-y','-r',str(MMROOT),'-n','amber26',
                    '-c','conda-forge','python=3.11','ambertools=26.0',
                    'parmed','pandas','matplotlib'],check=True)
else:
    print('AmberTools environment already installed.')
subprocess.run([str(EXE),'run','-r',str(MMROOT),'-n','amber26',
                'python','-c','import parmed; print("ParmEd OK")'],check=True)
print('AmberTools:',ENV/'bin'/'MMPBSA.py')

## 3. Install the corrected MM/GBSA driver

This writes the fixed driver locally; it does not overwrite completed output blocks in Drive.


In [ ]:
from pathlib import Path
import subprocess
DRIVER = Path('/content/cyp51_mmgbsa_driver.py')
DRIVER.write_text('#!/usr/bin/env python3\n"""CYP51 CAL028/CAL015 single-trajectory MM/GBSA, AmberTools / Google Drive.\nRead-only access to MD source files; output written to a separate directory.\n\nInvocation (in the AmberTools conda environment):\n  python cyp51_mmgbsa_driver.py --system CAL028 --mode smoke\n  python cyp51_mmgbsa_driver.py --system CAL028 --mode full\n  python cyp51_mmgbsa_driver.py --system CAL015 --mode full\n  python cyp51_mmgbsa_driver.py --system CAL028 --mode decomp\n  python cyp51_mmgbsa_driver.py --system CAL028 --mode report\n\nNo MM/GBSA values are predicted by this script: all values are parsed from\nsuccessful AmberTools MMPBSA.py output on the supplied DCD trajectories.\n"""\nimport argparse\nimport csv\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport numpy as np\nimport parmed as pmd\n\nDEFAULT_SOURCE = Path(\'/content/drive/MyDrive/CYP51_MD_animation_sources\')\nDEFAULT_OUTPUT = Path(\'/content/drive/MyDrive/CYP51_MD_MMGBSA\')\n\nSOLVENT_IONS = {\n    \'WAT\',\'HOH\',\'SOL\',\'TIP3\',\'TIP3P\',\'TIP4\',\'TIP4P\',\'OPC\',\'OPC3\',\'SPC\',\'SPCE\',\n    \'NA\',\'Na+\',\'SOD\',\'CL\',\'Cl-\',\'CLA\',\'K\',\'K+\',\'POT\',\'MG\',\'MG2\',\'Mg2+\',\'CA\',\'Ca2+\',\n}\n\nTERMS = [\'VDWAALS\',\'EEL\',\'EGB\',\'ESURF\',\'DELTA G gas\',\'DELTA G solv\',\'DELTA TOTAL\']\n\n\ndef exec_cmd(args, log_path, cwd=None):\n    args = [str(x) for x in args]\n    log_path = Path(log_path)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    print(\'$\', \' \'.join(args), flush=True)\n    with log_path.open(\'w\') as log:\n        p = subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE,\n                             stderr=subprocess.STDOUT, text=True, bufsize=1)\n        for line in p.stdout:\n            log.write(line)\n            log.flush()\n            if (\'WARNING\' in line.upper() or \'ERROR\' in line.upper() or\n                \'Processing\' in line or \'Timing\' in line or \'MMPBSA\' in line):\n                print(line.rstrip()[:250], flush=True)\n        code = p.wait()\n    if code:\n        tail=\'\\n\'.join(log_path.read_text(errors=\'replace\').splitlines()[-45:])\n        raise RuntimeError(f\'Command returned {code}. Log: {log_path}\\n{tail}\')\n    return log_path\n\n\ndef audit_topology(source):\n    parm = pmd.load_file(str(source))\n    names = [r.name for r in parm.residues]\n    ligand_res = [r for r in parm.residues if r.name == \'LIG\']\n    heme_res = [r for r in parm.residues if r.name == \'HEM\']\n    if len(ligand_res)!=1 or len(heme_res)!=1:\n        raise RuntimeError(f\'Expected exactly one LIG and one HEM residue; found \'\n                           f\'{len(ligand_res)} LIG, {len(heme_res)} HEM. \'\n                           \'Do not guess ligand masks.\')\n    cross_bonds=[]\n    for b in parm.bonds:\n        a1,a2=b.atom1,b.atom2\n        if (a1.residue.name == \'LIG\') != (a2.residue.name == \'LIG\'):\n            cross_bonds.append((str(a1), str(a2)))\n    if cross_bonds:\n        raise RuntimeError(f\'Ligand has cross-component covalent bonds: {cross_bonds[:5]}. \'\n                           \'Standard noncovalent MM/GBSA splitting is invalid.\')\n    heme_cys = [(str(b.atom1),str(b.atom2)) for b in parm.bonds\n                if ((b.atom1.residue.name == \'HEM\') != (b.atom2.residue.name == \'HEM\'))\n                and (\'HEM\' in (b.atom1.residue.name, b.atom2.residue.name))]\n    strip_names = sorted(set(names) & SOLVENT_IONS)\n    if not strip_names:\n        raise RuntimeError(\'No water/ion residues identified; inspect the topology before stripping.\')\n    expected_solute_atoms = sum(len(r.atoms) for r in parm.residues if r.name not in strip_names)\n    # Radius provenance: do not silently assume correct Fe solvation physics.\n    fe = [a for a in parm.atoms if a.residue.name==\'HEM\' and a.name.upper()==\'FE\']\n    fe_radii = [getattr(a, \'solvent_radius\', None) for a in fe]\n    meta = {\n        \'original_atoms\': len(parm.atoms),\n        \'original_residues\': len(parm.residues),\n        \'ligand_resid\': ligand_res[0].idx + 1,\n        \'ligand_atom_count\': len(ligand_res[0].atoms),\n        \'ligand_net_charge_e\': float(sum(a.charge for a in ligand_res[0].atoms)),\n        \'heme_resid\': heme_res[0].idx + 1,\n        \'heme_protein_cross_bonds\': heme_cys,\n        \'heme_FE_solvation_radii_A\': fe_radii,\n        \'strip_residue_names\': strip_names,\n        \'expected_dry_atoms\': expected_solute_atoms,\n        \'output_format\': \'kcal/mol\',\n        \'notes\': \'Heme included in the receptor; ligand remains its original charged form.\'\n    }\n    print(\'Source topology:\', source)\n    print(\'Atoms:\', len(parm.atoms), \'Dry complex atoms:\', expected_solute_atoms)\n    print(\'LIG residue:\', meta[\'ligand_resid\'], \'charge:\', round(meta[\'ligand_net_charge_e\'], 4))\n    print(\'HEM residue:\', meta[\'heme_resid\'], \'Fe radii:\', fe_radii)\n    print(\'Heme-protein bonds:\', heme_cys[:4])\n    print(\'Water/ion labels to strip:\', strip_names)\n    return meta\n\n\ndef prepare_topologies(system, source_root, out_root):\n    source = source_root / system / \'system.prmtop\'\n    if not source.is_file():\n        raise FileNotFoundError(source)\n    out = out_root/system\n    out.mkdir(parents=True, exist_ok=True)\n    auditfile=out/\'topology_audit.json\'\n    meta = audit_topology(source)\n    auditfile.write_text(json.dumps(meta,indent=2))\n    complexp,receptorp,ligandp=[out/name for name in (\'complex.prmtop\',\'receptor.prmtop\',\'ligand.prmtop\')]\n    if not all(p.is_file() and p.stat().st_size>1000 for p in (complexp,receptorp,ligandp)):\n        mask=\':\'+\',\'.join(meta[\'strip_residue_names\'])\n        cmd=[\'ante-MMPBSA.py\', \'-p\', source, \'-c\', complexp, \'-r\', receptorp,\n             \'-l\', ligandp, \'-s\', mask, \'-n\', \':LIG\', \'--radii=mbondi2\']\n        exec_cmd(cmd,out/\'ante_mmpbsa.log\',cwd=out)\n    comp,rec,lig=(pmd.load_file(str(p)) for p in (complexp,receptorp,ligandp))\n    if len(comp.atoms)!=meta[\'expected_dry_atoms\']:\n        raise RuntimeError(f\'Dry topology atom count {len(comp.atoms)} does not match \'\n                           f\'expected {meta["expected_dry_atoms"]}. Check solvent mask.\')\n    if len(rec.atoms)+len(lig.atoms)!=len(comp.atoms):\n        raise RuntimeError(\'Receptor + ligand atom totals differ from complex.\')\n    if any(r.name==\'HEM\' for r in lig.residues) or not any(r.name==\'HEM\' for r in rec.residues):\n        raise RuntimeError(\'HEM must remain in receptor, not the ligand.\')\n    if abs(sum(a.charge for a in lig.atoms)-meta[\'ligand_net_charge_e\'])>0.001:\n        raise RuntimeError(\'Ligand charge changed while preparing MM/GBSA topology.\')\n    # Some heme force fields produce missing PB radii, which should not be silently fixed.\n    bad=[]\n    for a in comp.atoms:\n        radius=getattr(a,\'solvent_radius\',None)\n        if radius is None or not np.isfinite(float(radius)) or float(radius)<=0:\n            bad.append((a.residue.name, a.name, radius))\n    if bad:\n        raise RuntimeError(f\'Invalid GB/SA radii for {len(bad)} atoms (first 12: \'\n                           f\'{bad[:12]}). Need validate heme Fe/nonstandard atom radii \'\n                           \'rather than claim an energy calculation with invalid parameters.\')\n    (out/\'topology_audit.json\').write_text(json.dumps({**meta,\n        \'dry_complex_atoms\':len(comp.atoms), \'receptor_atoms\':len(rec.atoms),\n        \'ligand_atoms\':len(lig.atoms),\'ligand_charge_after_strip\':float(sum(a.charge for a in lig.atoms)),\n        \'GB_model\':\'igb=5 (OBC2), mbondi2 radii, saltcon=0.150 M\'},indent=2))\n    print(f\'{system}: topology preparation and component checks PASSED.\',flush=True)\n    return meta\n\n\ndef block_dry(system, block, meta, source_root, out_root, smoke=False):\n    out=out_root/system/(\'smoke\' if smoke else f\'block_{block:02d}\')\n    out.mkdir(parents=True,exist_ok=True)\n    dry=out/\'sample_dry.nc\'\n    if dry.exists() and dry.stat().st_size>10000:\n        print(system, \'reusing prepared dry trajectory:\', dry)\n        return dry\n    src=source_root/system\n    begin=1 if smoke else (block-1)*10+1\n    end=2 if smoke else block*10\n    step=50 if smoke else 25   # 2 frames/ns test; 4 frames/ns for 50ns analysis\n    lines=[f\'parm {src/"system.prmtop"}\']\n    for ns in range(begin,end+1):\n        dcd=src/f\'ns_{ns:03d}.dcd\'\n        if not dcd.is_file(): raise FileNotFoundError(dcd)\n        lines.append(f\'trajin {dcd} 1 last {step}\')\n    lines += [\'autoimage\', \'strip :\'+\',\'.join(meta[\'strip_residue_names\']),\n              f\'trajout {dry} netcdf\', \'run\', \'quit\']\n    script=out/\'strip_and_sample.cpptraj\'\n    script.write_text(\'\\n\'.join(lines)+\'\\n\')\n    exec_cmd([\'cpptraj\',\'-i\',script],out/\'cpptraj.log\',cwd=out)\n    if not dry.exists() or dry.stat().st_size<10000:\n        raise RuntimeError(f\'cpptraj did not produce a valid-looking NetCDF: {dry}\')\n    # independent topology/trajectory validation with cpptraj itself\n    check=out/\'audit_dry.cpptraj\'\n    check.write_text(f\'parm {out_root/system/"complex.prmtop"}\\n\'\n                     f\'trajin {dry} 1 1\\nrun\\nquit\\n\')\n    exec_cmd([\'cpptraj\',\'-i\',check],out/\'audit_dry.log\',cwd=out)\n    print(system, \'prepared dry trajectory:\',dry, flush=True)\n    return dry\n\n\ndef gb_input(decomp=False):\n    general=\'\'\'&general\\n startframe=1,\\n endframe=999999,\\n interval=%s,\\n keep_files=0,\\n verbose=1,\\n/\\n\'\'\' % (4 if decomp else 1)\n    gb=\'\'\'&gb\\n igb=5,\\n saltcon=0.150,\\n/\\n\'\'\'\n    decompblock=\'\'\'&decomp\\n idecomp=2,\\n dec_verbose=1,\\n print_res="within 5",\\n/\\n\'\'\' if decomp else \'\'\n    return general+gb+decompblock\n\n\ndef parse_results(p):\n    s=Path(p).read_text(errors=\'replace\')\n    if \'INCONSISTENCIES EXIST\' in s.upper():\n        raise RuntimeError(f\'MMPBSA reported internal-energy inconsistencies. Inspect {p}\')\n    if \'GENERALIZED BORN:\' not in s:\n        raise RuntimeError(f\'No GENERALIZED BORN section found in {p}.\')\n    gb=s.split(\'GENERALIZED BORN:\',1)[1].split(\'POISSON BOLTZMANN:\',1)[0]\n    anchor=\'Differences (Complex - Receptor - Ligand):\'\n    if anchor not in gb:\n        raise RuntimeError(f\'No DELTA component section found in {p}.\')\n    delta=gb.split(anchor,1)[1]\n    result={}\n    num=r\'[-+]?(?:\\d*\\.\\d+|\\d+\\.?\\d*)(?:[Ee][-+]?\\d+)?\'\n    for name in TERMS:\n        m=re.search(r\'^\\s*\'+re.escape(name)+r\'\\s+(\'+num+r\')\\s+(\'+num+r\')\',delta,re.M)\n        if m:\n            result[name]={\'mean\':float(m.group(1)),\'sd\':float(m.group(2))}\n    if \'DELTA TOTAL\' not in result:\n        raise RuntimeError(f\'Missing delta total in {p}. The source file is retained.\')\n    return result\n\n\ndef run_one(system, block, meta, source_root, out_root, smoke=False, decomp=False):\n    bdir=out_root/system/(\'smoke\' if smoke else f\'block_{block:02d}\')\n    target_name = \'MMGBSA_DECOMP\' if decomp else \'MMGBSA\'\n    target=bdir/f\'{target_name}_summary.json\'\n    if target.is_file() and not smoke:\n        try:\n            data=json.loads(target.read_text())\n            if \'DELTA TOTAL\' in data[\'terms\']:\n                print(f\'{system} block {block}: already completed, skipping.\', flush=True)\n                return data\n        except (ValueError,KeyError): pass\n    dry=block_dry(system,block,meta,source_root,out_root,smoke=smoke)\n    infile=bdir/(\'mmpbsa_decomp.in\' if decomp else \'mmpbsa.in\')\n    infile.write_text(gb_input(decomp=decomp))\n    complexp,receptorp,ligandp=[out_root/system/name for name in (\'complex.prmtop\',\'receptor.prmtop\',\'ligand.prmtop\')]\n    prefix=\'DECOMP\' if decomp else \'GBSA\'\n    outfile=bdir/f\'{prefix}_RESULTS.dat\'\n    cmd=[\'MMPBSA.py\',\'-O\',\'-i\',infile,\'-cp\',complexp,\'-rp\',receptorp,\'-lp\',ligandp,\n         \'-y\',dry, \'-o\',outfile, \'-eo\',bdir/f\'{prefix}_perframe.csv\']\n    if decomp:\n        cmd+=[\'-do\',bdir/\'DECOMP_RESIDUES.dat\',\'-deo\',bdir/\'DECOMP_RESIDUES.csv\']\n    print(f\'\\n=== {system}: {"SMOKE" if smoke else f"{(block-1)*10}-{block*10} ns"}\'\n          f\' (GB-OBC2, {"decomposition" if decomp else "standard"}) ===\',flush=True)\n    scratch=Path(\'/content/CYP51_MMGBSA_scratch\')/system/(\'smoke\' if smoke else f\'block_{block:02d}\')\n    scratch.mkdir(parents=True,exist_ok=True)\n    exec_cmd(cmd,bdir/f\'{prefix}_run.log\',cwd=scratch)\n    terms=parse_results(outfile)\n    result={\'system\':system,\'block\':block,\'smoke\':smoke,\'decomposition\':decomp,\n            \'frames_expected\':(4 if smoke else 10 if decomp else 40),\n            \'time_ns\':[0,2] if smoke else [(block-1)*10,block*10],\n            \'terms\':terms,\'result_file\':str(outfile),\'trajectory\':str(dry),\n            \'calculation_model\':\'GB-OBC2 (igb 5), mbondi2, saltcon 0.15 M; no entropy term\',\n            \'datetime_utc\':datetime.now(timezone.utc).isoformat()}\n    target.write_text(json.dumps(result,indent=2))\n    print(f\'{system} block {block} ΔG_MMGBSA (no entropy): \'\n          f\'{terms["DELTA TOTAL"]["mean"]:.3f} ± {terms["DELTA TOTAL"]["sd"]:.3f} kcal/mol\',flush=True)\n    return result\n\n\ndef summarize(system,out_root):\n    """Create block summaries, including partial results; plotting is optional."""\n    import csv\n    from statistics import stdev\n    folder=out_root/system\n    results=[]\n    for block in range(1,6):\n        p=folder/f\'block_{block:02d}\'/\'MMGBSA_summary.json\'\n        if p.is_file():\n            results.append(json.loads(p.read_text()))\n    if not results:\n        print(f\'{system}: no completed 10ns blocks yet.\')\n        return\n    records=[]\n    for r in sorted(results,key=lambda x:x[\'block\']):\n        record={\'system\':system,\'block\':r[\'block\'],\n                \'midpoint_ns\':(r[\'block\']-0.5)*10,\n                \'n_frames\':r[\'frames_expected\']}\n        for key,value in r[\'terms\'].items():\n            record[key]=value[\'mean\']\n        records.append(record)\n    fields=[\'system\',\'block\',\'midpoint_ns\',\'n_frames\'] + TERMS\n    with (folder/\'MMGBSA_10ns_block_means.csv\').open(\'w\',newline=\'\') as handle:\n        writer=csv.DictWriter(handle,fieldnames=fields,extrasaction=\'ignore\')\n        writer.writeheader();writer.writerows(records)\n    frames=sum(r[\'n_frames\'] for r in records)\n    pooled={key:(sum(r[key]*r[\'n_frames\'] for r in records if key in r) /\n                 sum(r[\'n_frames\'] for r in records if key in r))\n            for key in TERMS if any(key in r for r in records)}\n    means=[r[\'DELTA TOTAL\'] for r in records]\n    pooled.update({\'n_completed_10ns_blocks\':len(records),\n                   \'n_analyzed_frames\':frames,\n                   \'block_to_block_SD_kcal_mol\':stdev(means) if len(means)>1 else None,\n                   \'block_mean_SEM_kcal_mol\':stdev(means)/len(means)**0.5 if len(means)>1 else None,\n                   \'full_50ns_covered\':len(records)==5})\n    (folder/\'MMGBSA_aggregate_summary.json\').write_text(json.dumps(pooled,indent=2))\n    try:\n        import matplotlib\n        matplotlib.use(\'Agg\')\n        import matplotlib.pyplot as plt\n        fig,ax=plt.subplots(figsize=(8,4.7))\n        ax.plot([r[\'midpoint_ns\'] for r in records],means,\'o-\',linewidth=1.8)\n        ax.set(xlim=(0,50),xlabel=\'Simulation time (ns)\',\n               ylabel=r\'MM/GBSA $\\Delta G$ (kcal/mol)\',\n               title=f\'{system}: MM/GBSA by 10 ns block\')\n        ax.grid(alpha=0.2)\n        fig.tight_layout()\n        fig.savefig(folder/\'MMGBSA_10ns_block_trend.png\',dpi=300)\n        plt.close(fig)\n    except ImportError:\n        print(\'Plot skipped (matplotlib unavailable); numerical reports are saved.\',flush=True)\n    print(f\'{system}: {len(records)} blocks, {frames} frames reported.\',flush=True)\n    if len(records)<5:\n        print(\'PARTIAL RESULT — 50 ns analysis not complete.\',flush=True)\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--system\',required=True,choices=[\'CAL028\',\'CAL015\'])\n    ap.add_argument(\'--mode\',required=True,choices=[\'smoke\',\'full\',\'report\',\'decomp\'])\n    ap.add_argument(\'--source-root\',default=str(DEFAULT_SOURCE))\n    ap.add_argument(\'--out-root\',default=str(DEFAULT_OUTPUT))\n    args=ap.parse_args()\n    source_root=Path(args.source_root)\n    out_root=Path(args.out_root)\n    if args.mode==\'report\':\n        summarize(args.system,out_root);return\n    for binary in (\'ante-MMPBSA.py\',\'cpptraj\',\'MMPBSA.py\'):\n        if not shutil.which(binary):\n            raise RuntimeError(f\'{binary} not found in PATH. Run in the dedicated AmberTools environment.\')\n    meta=prepare_topologies(args.system,source_root,out_root)\n    if args.mode==\'smoke\':\n        run_one(args.system,1,meta,source_root,out_root,smoke=True)\n    elif args.mode==\'full\':\n        for block in range(1,6):\n            run_one(args.system,block,meta,source_root,out_root)\n        print(f\'{args.system}: all five blocks completed; run --mode report separately.\', flush=True)\n    elif args.mode==\'decomp\':\n        for block in range(1,6):\n            run_one(args.system,block,meta,source_root,out_root,decomp=True)\n    print(\'Done. Source DCDs/topologies were not modified.\',flush=True)\n\nif __name__==\'__main__\':\n    main()\n')
subprocess.run(['python', '-m', 'py_compile', str(DRIVER)], check=True)
print('Corrected MM/GBSA driver installed:', DRIVER)

def run_mmgbsa(system, mode):
    cmd=[str(EXE),'run','-r',str(MMROOT),'-n','amber26',
         'python',str(DRIVER),'--system',system,'--mode',mode]
    print('Starting',system,mode,flush=True)
    subprocess.run(cmd,check=True)
    print('Finished',system,mode,flush=True)


## 4. CAL-028 smoke test already passed

Skip the next smoke-test cell if it already passed in the same Drive folder. It is provided only for a new environment validation.


In [ ]:
run_mmgbsa('CAL028','smoke')

## 5. CAL-015: run smoke test first

Execute this before starting production calculations for CAL-015.


In [ ]:
run_mmgbsa('CAL015','smoke')

## 6. CAL-028: run 5 resumable 10 ns blocks

Existing completed blocks are skipped. Generate summary after the full run.


In [ ]:
run_mmgbsa('CAL028','full')

## 7. CAL-015: MM/GBSA over full 50 ns

In [ ]:
run_mmgbsa('CAL015','full')

## 8. Generate reports and export

Numerical summaries are written even when matplotlib is unavailable. Graphs are optional.


In [ ]:
import json
OUT=Path('/content/drive/MyDrive/CYP51_MD_MMGBSA')
for system in ('CAL028','CAL015'):
    run_mmgbsa(system,'report')
    p=OUT/system/'MMGBSA_aggregate_summary.json'
    if p.is_file():
        d=json.loads(p.read_text())
        print('\n',system,'— completed blocks:',d.get('n_completed_10ns_blocks'))
        print('Mean DELTA TOTAL (kcal/mol):',d.get('DELTA TOTAL'))
        print('Full 50 ns:',d.get('full_50ns_covered'))
print('Results:',OUT)

## 9. Optional: residue-wise GB energy decomposition

Run only after the standard full analysis succeeds. Uses **10 sampled frames per 10 ns block** (50 frames/system, evenly spanning 50 ns). Outputs per-residue decomposition files but does not assume heme and protein residues have the same numbering as the original crystal structure. Mapping must be checked against the saved topology.

In [ ]:
# Uncomment these lines after standard MM/GBSA succeeds:
# run_mmgbsa('CAL028','decomp')
# run_mmgbsa('CAL015','decomp')
print('Residue decomposition remains optional; not executed in this cell.')

## 10. Package results for manuscript analysis and sharing

In [ ]:
import zipfile
OUT=Path('/content/drive/MyDrive/CYP51_MD_MMGBSA')
archive=OUT/'CYP51_MMGBSA_results_share.zip'
include={'.json','.csv','.dat','.png','.log','.in'}
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for system in ('CAL028','CAL015'):
        for p in sorted((OUT/system).rglob('*')):
            if p.is_file() and p.suffix.lower() in include and not p.name.startswith('_MMPBSA_'):
                z.write(p,p.relative_to(OUT))
print('Share this archive after both systems finish:',archive)
print('Archive size:',round(archive.stat().st_size/(1024**2),2),'MiB')

## Reporting details

- **Complex:** CYP51 + covalently attached HEM + LIG.
- **Receptor:** CYP51 + HEM (keep Fe–Cys bond).
- **Ligand:** LIG with its original formal charge, without neutralization or protonation changes.
- **GB model:** OBC2 (`igb=5`), `mbondi2` radii, 0.150 M salt. All energies in kcal/mol.
- **Frames:** 200 snapshots per candidate, 4/ns, taken over 0–50 ns, processed in 5 ten-ns blocks.
- **Reported terms:** ΔE_vdW, ΔE_elec, ΔG_GB, ΔG_SA, ΔTOTAL, and 10 ns trends.
- **Sampling limitations:** Do not treat frame-to-frame standard errors as independent biological replicates; the five block means indicate time dependence. No entropy term is included.
- **Heme model:** Fe–S and porphyrin bonding remain defined by the original IC6 topology; solvation radii and force field must be valid for the metal center. A failure is a preparation issue to address, not a result to discard silently.

When complete, upload the two `MMGBSA_aggregate_summary.json` files and their associated `MMGBSA_10ns_block_means.csv` files or the result package for manuscript drafting.